In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/home-credit-credit-risk-model-stability/sample_submission.csv
/kaggle/input/competitions/home-credit-credit-risk-model-stability/feature_definitions.csv
/kaggle/input/competitions/home-credit-credit-risk-model-stability/parquet_files/test/test_deposit_1.parquet
/kaggle/input/competitions/home-credit-credit-risk-model-stability/parquet_files/test/test_applprev_2.parquet
/kaggle/input/competitions/home-credit-credit-risk-model-stability/parquet_files/test/test_static_cb_0.parquet
/kaggle/input/competitions/home-credit-credit-risk-model-stability/parquet_files/test/test_static_0_0.parquet
/kaggle/input/competitions/home-credit-credit-risk-model-stability/parquet_files/test/test_credit_bureau_a_1_3.parquet
/kaggle/input/competitions/home-credit-credit-risk-model-stability/parquet_files/test/test_credit_bureau_a_1_2.parquet
/kaggle/input/competitions/home-credit-credit-risk-model-stability/parquet_files/test/test_tax_registry_b_1.parquet
/kaggle/input/competitions

读取基本的特征定义

In [3]:
PATH = "/kaggle/input/competitions/home-credit-credit-risk-model-stability/"

feature_def = pd.read_csv(PATH + "feature_definitions.csv")

feature_def.head(20)

,Variable,Description
0,actualdpd_943P,Days Past Due (DPD) of previous contract (actu...
1,actualdpdtolerance_344P,DPD of client with tolerance.
2,addres_district_368M,District of the person's address.
3,addres_role_871L,Role of person's address.
4,addres_zip_823M,Zip code of the address.
5,amount_1115A,Credit amount of the active contract provided ...
6,amount_416A,Deposit amount.
7,amount_4527230A,Tax deductions amount tracked by the governmen...
8,amount_4917619A,Tax deductions amount tracked by the governmen...
9,amtdebitincoming_4809443A,Incoming debit card transactions amount.


### 1.1 物理分片

目的：把所有表的"schema 清单"扫出来，确认数据的物理结构：有哪些文件、各自多大、包含哪些列。

方法：由于数据总量在 GB 级，单机内存无法一次性容纳，此处使用 Polars 的惰性接口
（`scan_parquet`）只读取 Parquet 文件头中的元数据，不加载实际数据行

产出：`info` 字典，记录每个文件的行数与列名清单，供下一步归并分片使用。

In [4]:
import polars as pl, pathlib
from collections import defaultdict

root = pathlib.Path("/kaggle/input/competitions/home-credit-credit-risk-model-stability/parquet_files/train")
print(root.exists(), len(list(root.glob("*.parquet"))))   # 先确认路径存在、文件数对得上

info = {}
for f in sorted(root.glob("*.parquet")):
    lf = pl.scan_parquet(f)                  # 惰性：只读文件头元数据，不加载数据行
    cols = lf.collect_schema().names()       # 列名存在 parquet 文件尾，取它无需扫数据
    n = lf.select(pl.len()).collect().item() # 行数同理，优化器直接读 row-group 统计
    info[f.stem] = {"n_rows": n, "cols": cols}
    print(f"{f.stem:32s} rows={n:>10,}  cols={len(cols)}")

True 32
train_applprev_1_0               rows= 3,887,684  cols=41
train_applprev_1_1               rows= 2,638,295  cols=41
train_applprev_2                 rows=14,075,487  cols=6
train_base                       rows= 1,526,659  cols=5
train_credit_bureau_a_1_0        rows= 4,108,212  cols=79
train_credit_bureau_a_1_1        rows= 6,009,192  cols=79
train_credit_bureau_a_1_2        rows= 3,743,810  cols=79
train_credit_bureau_a_1_3        rows= 2,079,323  cols=79
train_credit_bureau_a_2_0        rows= 5,296,031  cols=19
train_credit_bureau_a_2_1        rows= 7,861,809  cols=19
train_credit_bureau_a_2_10       rows= 4,386,062  cols=19
train_credit_bureau_a_2_2        rows=17,893,536  cols=19
train_credit_bureau_a_2_3        rows=26,563,901  cols=19
train_credit_bureau_a_2_4        rows=27,025,737  cols=19
train_credit_bureau_a_2_5        rows=33,053,760  cols=19
train_credit_bureau_a_2_6        rows=25,511,332  cols=19
train_credit_bureau_a_2_7        rows= 8,055,986  cols=19
train_cr

### 1.2 物理分片 ——> 逻辑表

目的：上一步扫出 32 个文件，但它们不对应 32 张表——大表被切成了多个分片。后续分析需以逻辑表
为单位，故先归并。

方法：按列名集合归并，而非文件名。文件名中的数字含义不统一——
`train_static_0_0` 末位是分片号，`train_static_cb_0` 末位是深度，用正则拆解
会误判；而同一逻辑表的分片必然共享完全相同的列集合，此判据无歧义。

产出：逻辑表清单，含分片文件名、合计行数、主键列。主键列直接给出表的粒度：
`case_id` 为 depth 0（可直接 join），加 `num_group1` 为 depth 1（需聚合），
再加 `num_group2` 为 depth 2（需聚合两次）。

In [5]:
groups = defaultdict(list)
for name, v in info.items():
    groups[tuple(v["cols"])].append(name)

for cols, files in sorted(groups.items(), key=lambda x: -len(x[1])):
    rows = sum(info[f]["n_rows"] for f in files)
    print(f"{len(files):>2} shard(s)  rows={rows:>12,}  cols={len(cols):>3}")
    print("     ", files)

11 shard(s)  rows= 188,298,452  cols= 19
      ['train_credit_bureau_a_2_0', 'train_credit_bureau_a_2_1', 'train_credit_bureau_a_2_10', 'train_credit_bureau_a_2_2', 'train_credit_bureau_a_2_3', 'train_credit_bureau_a_2_4', 'train_credit_bureau_a_2_5', 'train_credit_bureau_a_2_6', 'train_credit_bureau_a_2_7', 'train_credit_bureau_a_2_8', 'train_credit_bureau_a_2_9']
 4 shard(s)  rows=  15,940,537  cols= 79
      ['train_credit_bureau_a_1_0', 'train_credit_bureau_a_1_1', 'train_credit_bureau_a_1_2', 'train_credit_bureau_a_1_3']
 2 shard(s)  rows=   6,525,979  cols= 41
      ['train_applprev_1_0', 'train_applprev_1_1']
 2 shard(s)  rows=   1,526,659  cols=168
      ['train_static_0_0', 'train_static_0_1']
 1 shard(s)  rows=  14,075,487  cols=  6
      ['train_applprev_2']
 1 shard(s)  rows=   1,526,659  cols=  5
      ['train_base']
 1 shard(s)  rows=      85,791  cols= 45
      ['train_credit_bureau_b_1']
 1 shard(s)  rows=   1,286,755  cols=  6
      ['train_credit_bureau_b_2']
 1 shard

### 1.3 验证层级键

目的：确认两件事——一行是否真的对应你以为的那个东西（一个客户 / 一次申请 /
一条明细），以及子表里出现的客户在主表中是否都能找到。这是后续聚合与 join 的前提。

方法：对候选键做两项检验。

- **唯一性**：`rows == unique(key)` 是否成立。成立则该键组合确为该表主键，
  粒度假设正确；不成立说明还有未识别的层级。
- **覆盖率**：子表的 `case_id` 有多大比例能在 base 中找到。为 1 说明左连接不丢数据；偏低则存在孤儿记录，需查明原因。

17 张逻辑表仅对应 3 种粒度模式，同深度表结构同构，故按模式抽样验证而非逐表验证（depth 1 取 `applprev_1`，depth 2 取 `applprev_2`）。未验表的覆盖率检查推迟到聚合之后进行——彼时表已压至客户级，同一检验的代价从扫描上亿行降至秒级。

需注意：左连接会静默丢弃对不上的记录，join 后的行数和 null 比例均无法反映此类丢失，故该检查必须显式做，不能指望它自行暴露。

产出：确认三种粒度模式成立——depth 0 一行一客户、depth 1 一行一次申请、
depth 2 一行一条申请明细；且子表客户全部能在 base 中找到，join 不丢数据。

In [6]:
base = pl.scan_parquet(root / "train_base.parquet")            # 主表：一行一客户
lf   = pl.scan_parquet(root / "train_applprev_1_*.parquet")    # 子表：一行一次申请

n_rows = lf.select(pl.len()).collect().item()                              # 总行数
n_uniq = lf.select("case_id", "num_group1").unique().collect().height      # 不重复的键组合数
n_ids  = lf.select("case_id").unique().collect().height                    # 涉及多少客户
n_hit  = (lf.select("case_id").unique()
            .join(base.select("case_id"), on="case_id", how="semi")        # 只留能在主表配上的
            .collect().height)

print(f"rows={n_rows:,}  唯一={n_rows == n_uniq}  覆盖率={n_hit / n_ids:.4f}")

rows=6,525,979  唯一=True  覆盖率=1.0000


In [7]:
lf2 = pl.scan_parquet(root / "train_applprev_2.parquet")     # depth 2：一行一条明细

n_rows = lf2.select(pl.len()).collect().item()
n_uniq = lf2.select("case_id", "num_group1", "num_group2").unique().collect().height
n_ids  = lf2.select("case_id").unique().collect().height
n_hit  = (lf2.select("case_id").unique()
             .join(base.select("case_id"), on="case_id", how="semi")
             .collect().height)

print(f"rows={n_rows:,}  唯一={n_rows == n_uniq}  覆盖率={n_hit / n_ids:.4f}")

rows=14,075,487  唯一=True  覆盖率=1.0000
